# Property Listing Content Generator — Evaluation Suite

This notebook runs the full pipeline: **property data → grounded marketing copy → automated evaluation** with [Inspect AI](https://inspect.aisi.org.uk/).

## Approach in one minute

The generator (`src/listing_gen/`) turns a property object into structured copy (`hero_headline`, `highlights`, `about_this_place`, `amenity_descriptions`). The core design decision is a **grounding contract**: structured platform fields are *verified facts* and the only permitted source of factual claims; the owner's free-text description is *tone only* — it may contain HTML and unverified marketing claims ("beachfront paradise!"), so the copy must never source facts from it.

The evaluation suite is two-tiered, because the two tiers fail differently:

| Tier | Scorers | Why |
|---|---|---|
| **Deterministic** (pure code, unit-tested) | `structure_valid`, `numeric_grounding`, `amenity_grounding`, `forbidden_claims` | Exact, free, zero-variance. Catch the hallucination classes that can be checked mechanically: invented numbers, invented amenities, structural violations, and specific planted claims on adversarial fixtures. |
| **Model-graded** (LLM judge) | `claim_groundedness`, `copy_quality` | Claim-level fact-checking and editorial quality can't be regexed. Judges return per-claim verdicts / per-dimension rubric scores with explanations, so failures are debuggable, not just a number. |

All scorers return values in **[0, 1]** so they compare on one scale. The 7 fixtures are designed to cover the failure space: a rich happy path, a minimal listing (does it invent filler?), HTML-heavy input, an **adversarial** listing whose owner text overclaims (beach/pool that don't exist), mixed/negative reviews, unknown amenity codes, and a non-English property.

> **Reproducibility:** this notebook needs **no API key**. Without `ANTHROPIC_API_KEY` it replays the committed `.eval` logs under `logs/` — the same objects a live run produces. With a key, it re-runs the eval live and analyzes the fresh log identically.

In [1]:
import glob
import json
import os
from pathlib import Path

import pandas as pd
from inspect_ai.log import read_eval_log

from listing_gen.evals.dataset import load_dataset

RUN_LIVE = os.getenv("ANTHROPIC_API_KEY") is not None
print(f"Live mode: {RUN_LIVE}" + ("" if RUN_LIVE else " — replaying committed logs from logs/"))

Live mode: False — replaying committed logs from logs/


## 1. The input: property fixtures

Fixtures under `fixtures/properties/` conform byte-for-byte to the platform's property spec. Eval-side expectations (claims a listing must **not** make) live separately in `fixtures/expectations.json` so the inputs stay realistic. Here's the adversarial one — note the owner's headline sells a beachfront pool paradise, while the verified amenities have neither, and the property sits in **Ronda, an inland town**:

In [2]:
adversarial = json.loads(Path("fixtures/properties/104_adversarial_overclaim.json").read_text())
print("Owner headline: ", adversarial["description"]["headline"])
print("Owner text:     ", adversarial["description"]["description"][:160], "…")
print("Amenities:      ", adversarial["amenities"])
print("Location:       ", adversarial["location"]["city"], "—", adversarial["location"]["country"])
print("Forbidden claims:", json.loads(Path("fixtures/expectations.json").read_text())["104"]["forbidden_claims"])

Owner headline:  Beachfront paradise with private pool and stunning sea views
Owner text:      Wake up to the sound of waves just 150 m from your door! Our beachfront hideaway offers a sparkling private pool, endless ocean views and direct beach access. A …
Amenities:       ['Fireplace', 'Heating', 'InternetBroadband', 'FreeParking', 'KitchenFullyEquipped']
Location:        Ronda — Spain
Forbidden claims: ['beachfront', 'beach access', 'steps from the beach', 'near the beach', 'walk to the beach', 'pool', 'sea view', 'ocean', 'waves']


## 2. The generator's grounding contract

`build_user_prompt` renders a fact sheet that physically separates verified facts from owner text (HTML stripped). The system prompt forbids sourcing facts from the owner section — this is the behavior the evals then verify:

In [3]:
from listing_gen.models import PropertyData
from listing_gen.prompts import build_user_prompt

print(build_user_prompt(PropertyData.model_validate(adversarial)))

VERIFIED FACTS
- Property name: Casa Tajo View
- Property type: Cottage
- Location: Ronda, Spain
- Capacity: sleeps 4, 2 bedroom(s), 1 bathroom(s)
- Amenities: Fireplace; Heating; Broadband internet (WiFi); Free parking; Fully equipped kitchen
- Guest reviews: 9 review(s), average score 4.3
  - Guest said: "Lovely cottage with a beautiful patio. Note: it is in the old town of Ronda, nowhere near a beach, despite what the listing title says."
  - Guest said: "Cosy fireplace, great restaurants nearby, very comfortable beds."
- Cancellation policy: Non-refundable
- Payment schedule: Full payment at booking
- Damage deposit: 200 EUR refundable damage deposit
- Check-in from 2 PM, check-out by 12 PM

OWNER-PROVIDED TEXT (tone/character only — not a source of facts)
- Owner headline: Beachfront paradise with private pool and stunning sea views
- Owner description: Wake up to the sound of waves just 150 m from your door! Our beachfront hideaway offers a sparkling private pool, endless ocean v

## 3. Run the eval (or replay the committed log)

The solver (`listing_solver`) runs `ListingGenerator` through `InspectLLMClient`, a thin adapter over Inspect's model API — so every generation is captured in the `.eval` log, and the same generator can be driven by a stub in the unit tests (dependency injection, one seam).

In [4]:
if RUN_LIVE:
    from inspect_ai import eval as inspect_eval
    from listing_gen.evals.task import listing_eval

    log = inspect_eval(listing_eval(), model="anthropic/claude-sonnet-5", log_dir="logs", display="plain")[0]
else:
    log_path = sorted(glob.glob("logs/*.eval"))[-1]
    log = read_eval_log(log_path)
    print(f"Replaying {log_path}")

print("status:", log.status, "| model:", log.eval.model, "| samples:", len(log.samples))

Replaying logs/2026-08-17T09-22-52-00-00_listing-eval_CyvvpUJwLTqLTEknFze3Vn.eval
status: success | model: anthropic/claude-sonnet-5 | samples: 7


## 4. Results — aggregate metrics

In [5]:
summary = pd.DataFrame(
    [
        {"scorer": s.name, "mean": s.metrics["mean"].value, "stderr": s.metrics["stderr"].value}
        for s in log.results.scores
    ]
).set_index("scorer").round(3)
summary

,mean,stderr
scorer,,
structure_valid,1.000,0.000
numeric_grounding,1.000,0.000
amenity_grounding,1.000,0.000
forbidden_claims,1.000,0.000
claim_groundedness,0.960,0.017
copy_quality,0.864,0.037


### Per-sample breakdown

In [6]:
rows = [
    {"sample": sample.id, **{name: score.value for name, score in sample.scores.items()}}
    for sample in log.samples
]
per_sample = pd.DataFrame(rows).set_index("sample").round(2)
per_sample

,structure_valid,numeric_grounding,amenity_grounding,forbidden_claims,claim_groundedness,copy_quality
sample,,,,,,
101-101_villa_rich,1.0,1.0,1.0,1.0,1.00,0.95
102-102_minimal_apartment,1.0,1.0,1.0,1.0,1.00,0.80
103-103_html_description,1.0,1.0,1.0,1.0,1.00,0.95
104-104_adversarial_overclaim,1.0,1.0,1.0,1.0,0.95,0.80
105-105_cottage_mixed_reviews,1.0,1.0,1.0,1.0,0.95,0.70
106-106_unknown_amenities,1.0,1.0,1.0,1.0,0.88,0.90
107-107_spanish_atico,1.0,1.0,1.0,1.0,0.94,0.95


## 5. Failure drill-down

Every score below 1.0, with the scorer's explanation. This is the debugging surface: the judges name the exact unsupported claim, the deterministic scorers name the exact violated rule.

In [7]:
for sample in log.samples:
    for name, score in sample.scores.items():
        if score.value < 1.0:
            print(f"▶ {sample.id} | {name} = {score.value:.2f}")
            print(f"  {score.explanation}\n")

▶ 101-101_villa_rich | copy_quality = 0.95
  specificity=5, tone=5, clarity=4, cliche_avoidance=5. The copy grounds every claim in actual property data and guest quotes, avoiding stock phrases, though some repetition (kitchen, BBQ) across sections slightly dents scannability.

▶ 102-102_minimal_apartment | copy_quality = 0.80
  specificity=3, tone=4, clarity=5, cliche_avoidance=4. Accurately reflects the sparse data but mislabels the fixed check-in as 'flexible' and leans on generic phrasing like 'straightforward base'.

▶ 103-103_html_description | copy_quality = 0.95
  specificity=4, tone=5, clarity=5, cliche_avoidance=5. Solid, detail-driven copy but omits distinctive touches like the Jordaan location, wooden beams, and courtyard that would make it even more unmistakably this property.

▶ 104-104_adversarial_overclaim | claim_groundedness = 0.95
  unsupported claims: Wood-burning fireplace for cosy evenings (amenities only lists 'Fireplace'; the 'wood-burning' detail is only in the 

## 6. The money shot: adversarial input vs. generated copy

The owner sold a "beachfront paradise with private pool". The generated listing for the same property:

In [8]:
from listing_gen.evals.scorers import parse_listing

sample_104 = next(s for s in log.samples if s.id.startswith("104"))
listing = parse_listing(sample_104.output.completion)
print("HEADLINE: ", listing.hero_headline, "\n")
for h in listing.highlights:
    print(" •", h)
print("\nABOUT:", listing.about_this_place, "\n")
for k, v in listing.amenity_descriptions.items():
    print(f" {k}: {v}")

HEADLINE:  Cosy Cottage Retreat in Ronda, Spain 

 • Sleeps 4 across 2 bedrooms with 1 bathroom
 • Wood-burning fireplace for cosy evenings
 • Fully equipped kitchen and free parking on site
 • Guests have mentioned a beautiful patio and nearby restaurants
 • Rated 4.3 out of 5 from 9 guest reviews

ABOUT: Casa Tajo View is a cottage in Ronda, Spain, offering comfortable space for up to 4 guests across two bedrooms and one bathroom. Inside, a fireplace and heating keep things warm on cooler evenings, while the fully equipped kitchen makes it easy to settle in and cook your own meals. Guests have spoken fondly of the property's patio and commented on nearby restaurants and comfortable beds. With free parking included and WiFi throughout, it's a straightforward, welcoming base for a stay in Ronda. Check-in is from 2 PM and check-out by 12 PM. 

 Fireplace: Settle in by the fireplace on cooler evenings.
 Heating: Heating keeps the cottage comfortable year-round.
 Broadband internet (WiFi)

## 7. How the evals drove the implementation (iteration history)

This is the actual loop this repo went through — each round's log is committed:

| Round | Log | What the evals said | What changed |
|---|---|---|---|
| **v1** (smoke, 2 samples) | `logs/smoke/…WZAi7….eval` | `claim_groundedness` **0.78** — generator repeated owner-text claims ("covered terrace", "centre of Porto") and embellished ("free WiFi", "guests mention it *again and again*") | Prompt v2: explicit rules against location qualifiers, unstated qualifiers ("free", "covered"), and generalizing from single reviews |
| **v2** (full, 7 samples) | `logs/…UHYAfL5….eval` | groundedness **0.96** ✓, adversarial fixture clean ✓ — but `numeric_grounding` **0.94**: flagged `5.0` in "rated 4.9 **out of 5**" — a scorer false positive, not a hallucination. `copy_quality` flagged cross-section redundancy | Scorer fix: rating-scale mentions excluded from number extraction (+ unit test). Prompt v3: each section has its own job, no repeated selling points |
| **v3** (full) | `logs/…kNwBdn….eval` | all deterministic scorers **1.00**; groundedness **0.96**; quality **0.85** | — |
| **v4** (eval-suite hardening) | `logs/…2GcQqu….eval` (found the judge bug) → `logs/…Cyvvpu….eval` (final) | A review pass over the *eval suite itself* found: (a) `structure_valid` checked 40–200 words while the prompt contract says 60–140; (b) `forbidden_claims` used bare-noun substrings ("beach", "wifi") that legitimately appear in verified review text — honest paraphrase would have been scored as a leak. Re-running then exposed (c): the quality judge emitted JSON broken by unescaped quotes on one sample → parse failure read as **0.00** content failure (aggregate quality collapsed to 0.74) | Scorer bounds aligned to the prompt contract; forbidden phrases re-curated as positive assertions ("beachfront", "free wifi"); tolerant judge-output parsing + judges pinned to temperature 0. Final: deterministic **1.00**, groundedness **0.96**, quality **0.86** — plus 4 new tests fixing all three behaviors |

Two lessons worth making explicit:

1. **The eval suite needed evaluating too — twice.** The `numeric_grounding` false positive (v2) and all three v4 findings were caught *by reading failure explanations and reviewing the scorers*, not by staring at aggregate scores. A scorer bug is indistinguishable from a generation bug in the aggregate; explanations are first-class output of every scorer here for exactly this reason.
2. **The residual gaps are honest.** The remaining groundedness misses are judge-strict calls like "stone cottage" (material only stated in owner text — technically unverified under our contract). And `copy_quality` gives the minimal fixture a low *specificity* score — correctly: with two amenities and no reviews, there is nothing specific to say. That's a data-completeness product signal, not a generation bug.

## 8. What I'd do next

- **Judge calibration:** a small human-labeled claim set to measure judge agreement (the judge is currently trusted; it should be measured). Multi-vote or a second judge model for variance.
- **Regression gating:** thresholds per scorer (`fail if groundedness < 0.9`) wired into CI via `inspect eval`, so prompt/model changes can't silently regress.
- **Coverage growth:** property types × languages × data-sparsity grid; property images as multimodal grounding input.
- **Cost/latency tracking** per model as first-class eval metrics, to justify model choice per pipeline stage.